In [82]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [83]:
BASE_URL = "https://api.deepseek.com"
DEEPSEEK_API_KEY = os.environ["DEEPSEEK_API_KEY"]
SILICONFLOW_API_KEY = os.environ["SILICONFLOW_API_KEY"]
deepseek_chat_model = "deepseek-chat"
deepseek_reason_model = "deepseek-reasoner"

# Naive RAG

In [84]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [85]:
loader = PyPDFLoader("knowledge_base.pdf")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
documents = text_splitter.split_documents(documents)

In [86]:
embedding = OpenAIEmbeddings(
    base_url="https://api.siliconflow.com/v1",
    api_key=SILICONFLOW_API_KEY,
    model="Qwen/Qwen3-Embedding-0.6B"
)

In [87]:
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedding,
    persist_directory="./chroma_db"
)

In [88]:
documents

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:4177c2c)', 'creationdate': '', 'author': 'Jian Yang; Xianglong Liu; Weifeng Lv; Ken Deng; Shawn Guo; Lin Jing; Yizhi Li; Shark Liu; Xianzhen Luo; Yuyu Luo; Changzai Pan; Ensheng Shi; Yingshui Tan; Renshuai Tao; Jiajun Wu; Xianjie Wu; Zhenhe Wu; Daoguang Zan; Chenchen Zhang; Wei Zhang; He Zhu; Terry Yue Zhuo; Kerui Cao; Xianfu Cheng; Jun Dong; Shengjie Fang; Zhiwei Fei; Xiangyuan Guan; Qipeng Guo; Zhiguang Han; Joseph James; Tianqi Luo; Renyuan Li; Yuhang Li; Yiming Liang; Congnan Liu; Jiaheng Liu; Qian Liu; Ruitong Liu; Tyler Loakman; Xiangxin Meng; Chuang Peng; Tianhao Peng; Jiajun Shi; Mingjie Tang; Boyang Wang; Haowen Wang; Yunli Wang; Fanglin Xu; Zihan Xu; Fei Yuan; Ge Zhang; Jiayi Zhang; Xinhao Zhang; Wangchunshu Zhou; Hualei Zhu; King Zhu; Bryan Dai; Aishan Liu; Zhoujun Li; Chenghua Lin; Tianyu Liu; Chao Peng; Kai Shen; Libo Qin; Shuangyong Song; Zizheng Zhan; Jiajun Zhang; Jie Zhang; Zhaoxiang Zh

In [89]:
from langchain_openai import ChatOpenAI
from langchain_deepseek import ChatDeepSeek
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate

In [90]:
vector_store = Chroma(persist_directory="./chroma_db", embedding_function=embedding)
query = "What is Code Large Language Model?"
docs = vector_store.similarity_search(query, k=3)
context = "\n\n".join([doc.page_content for doc in docs])

In [91]:
context

'Peng, Zhenghan Li, Jiahui Feng, Xiao Wei, et al. Large language model for verilog code\ngeneration: Literature review and the road ahead. 2025.\n1176 Jian Yang, Shuming Ma, Haoyang Huang, Dongdong Zhang, Li Dong, Shaohan Huang,\nAlexandre Muzio, Saksham Singhal, Hany Hassan, Xia Song, and Furu Wei. Multilingual\nmachine translation systems from microsoft for WMT21 shared task. In Loïc Barrault,\nOndrej Bojar, Fethi Bougares, Rajen Chatterjee, Marta R. Costa-jussà, Christian Federmann,\n\nPeng, Zhenghan Li, Jiahui Feng, Xiao Wei, et al. Large language model for verilog code\ngeneration: Literature review and the road ahead. 2025.\n1176 Jian Yang, Shuming Ma, Haoyang Huang, Dongdong Zhang, Li Dong, Shaohan Huang,\nAlexandre Muzio, Saksham Singhal, Hany Hassan, Xia Song, and Furu Wei. Multilingual\nmachine translation systems from microsoft for WMT21 shared task. In Loïc Barrault,\nOndrej Bojar, Fethi Bougares, Rajen Chatterjee, Marta R. Costa-jussà, Christian Federmann,\n\npairs plus re

In [92]:
prompt_template = ChatPromptTemplate(
   [
    ("ai", "You are a helpful assistant."),
    ("human", "Context: {context}\nQuestion: {question}\nAnswer:")
   ]
)
prompt_template.input_variables

['context', 'question']

In [93]:
llm = ChatOpenAI(
    base_url=BASE_URL, 
    api_key=DEEPSEEK_API_KEY, 
    model=deepseek_chat_model,
    temperature=0,
    max_retries=3
)
chain = prompt_template | llm
answer = chain.invoke({"context": "context", "question": {query}})

In [94]:
answer.pretty_print()

================================== Ai Message ==================================

A **Code Large Language Model (Code LLM)** is a specialized type of large language model (LLM) that is trained primarily on source code and programming-related text. Its main purpose is to understand, generate, modify, and explain code across various programming languages.

Key characteristics and capabilities include:

1. **Code Generation** – Writing code snippets or entire functions based on natural language prompts (e.g., "write a Python function to sort a list").
2. **Code Completion** – Suggesting the next lines or tokens of code in an integrated development environment (IDE).
3. **Code Explanation** – Translating code into plain English or another natural language.
4. **Debugging & Error Fixing** – Identifying bugs, suggesting fixes, or explaining error messages.
5. **Code Translation** – Converting code from one programming language to another (e.g., Python to JavaScript).
6. **Documentation Gener

# Agentic RAG

In [ ]:
from dataclasses import dataclass
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

In [ ]:
class KbController:
    def __init__(self, pdf_path: str, persist_dir: str = "./chroma_db"):
        self.pdf_path = pdf_path
        self.persist_dir = persist_dir
        self.file_registry = {}

        self.embeddings = embedding
        
        self.vectorstore = self._init_or_load_db()

    def _init_or_load_db(self) -> Chroma:
        # 为了演示，我们将 pdf 作为 file_id = 42 录入
        file_id = 42
        filename = os.path.basename(self.pdf_path)

        # 检查是否已存在持久化数据
        if os.path.exists(self.persist_dir) and os.listdir(self.persist_dir):
            vectorstore = Chroma(
                persist_directory=self.persist_dir, 
                embedding_function=self.embeddings
            )
            # 简单恢复文件注册表元数据（实际生产中应存入关系型数据库）
            # 这里简单假定 chunk_count 为向量库中文档总数
            self.file_registry[file_id] = {
                "id": file_id,
                "filename": filename,
                "chunk_count": len(vectorstore.get()["ids"]), 
                "status": "done"
            }
            return vectorstore

        # 如果不存在，则读取PDF并构建
        if not os.path.exists(self.pdf_path):
            raise FileNotFoundError(f"未找到文件: {self.pdf_path}")

        loader = PyPDFLoader(self.pdf_path)
        docs = loader.load()
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        chunks = text_splitter.split_documents(docs)
        
        self.file_registry[file_id] = {
            "id": file_id,
            "filename": filename,
            "chunk_count": len(chunks),
            "status": "done"
        }
        
        # 为每个 chunk 注入定位元数据
        for i, chunk in enumerate(chunks):
            chunk.metadata.update({
                "file_id": file_id,
                "chunk_index": i,
                "status": "done"
            })
        
        vectorstore = Chroma.from_documents(
            documents=chunks, 
            embedding=self.embeddings, 
            persist_directory=self.persist_dir
        )
        return vectorstore

    # --- 对应四个工具的核心接口 ---

    def search(self, kb_id: int, query: str) -> List[Dict[str, Any]]:
        # 相似度检索，返回最相关的几个分块元数据及简要内容
        docs = self.vectorstore.similarity_search(query, k=3)
        return [{"content": d.page_content[:100] + "...", "metadata": d.metadata} for d in docs]

    def getFilesMeta(self, kb_id: int, file_ids: List[int]) -> List[Dict[str, Any]]:
        return [self.file_registry[fid] for fid in file_ids if fid in self.file_registry]

    def readFileChunks(self, kb_id: int, chunks: List[Dict[str, int]]) -> List[Dict[str, Any]]:
        results = []
        for req in chunks:
            # 兼容不同命名习惯，获取 file_id 和 chunk_index
            fid = req.get("file_id", req.get("id"))
            idx = req.get("chunk_index")
            if fid is None or idx is None:
                continue
            
            # 使用 Chroma 的过滤语法精确查询
            res = self.vectorstore.get(
                where={"$and": [{"file_id": fid}, {"chunk_index": idx}]}
            )
            
            if res and res["documents"]:
                results.append({
                    "file_id": fid,
                    "chunk_index": idx,
                    "content": res["documents"][0]
                })
        return results

    def listFilesPaginated(self, kb_id: int, page: int, page_size: int) -> List[Dict[str, Any]]:
        files = list(self.file_registry.values())
        start = (page - 1) * page_size
        return files[start:start + page_size]


# ==========================================
# 2. 实例化控制器与 Agent 组装
# ==========================================

# 实例化真实的 Controller (请确保当前目录下有 knowledge_base.pdf)
kb_controller = KbController(pdf_path="knowledge_base.pdf")
knowledge_base_id = 42

@tool("query_knowledge_base")
def query_knowledge_base(query: str) -> Any:
    """Query a knowledge base. Returns relevant chunks with their metadata (including chunk_index)."""
    return kb_controller.search(knowledge_base_id, query)

@tool("get_files_meta")
def get_files_meta(fileIds: List[int]) -> Any:
    """Get metadata for files in the current knowledge base."""
    if not fileIds:
        return "Please provide an array of file IDs."
    return kb_controller.getFilesMeta(knowledge_base_id, fileIds)

@tool("read_file_chunks")
def read_file_chunks(chunks: List[Dict[str, int]]) -> Any:
    """
    Read full content chunks from specified files. 
    Input format: [{"file_id": 42, "chunk_index": 0}]
    """
    if not chunks:
        return "Please provide an array of chunks to read."
    return kb_controller.readFileChunks(knowledge_base_id, chunks)

@tool("list_files")
def list_files(page: int, pageSize: int) -> Any:
    """List all files in the current knowledge base. Returns file ID, filename, and chunk count for each file."""
    files = kb_controller.listFilesPaginated(knowledge_base_id, page, pageSize)
    return [
        {"id": f["id"], "filename": f["filename"], "chunkCount": f.get("chunk_count", 0)}
        for f in files
        if f.get("status") == "done"
    ]

# 工具清单
tools = [query_knowledge_base, get_files_meta, read_file_chunks, list_files]

# 行为策略（系统提示）
SYSTEM_PROMPT = """你是一个 Agentic RAG 助手。请遵循：
- 先用 query_knowledge_base 搜索；必要时使用 get_files_meta 查看文件信息，或用 list_files 浏览备选。
- 最终必须用 read_file_chunks 读取少量最相关的片段（根据搜索返回的 file_id 和 chunk_index），再基于片段内容作答。
- 不要编造；若证据不足请说明不足并建议下一步。
- 回答末尾用“引用：”列出你实际读取过的 fileId 和 chunkIndex（或文件名）。
"""

# 确保读取环境变量中的 API 密钥
api_key = os.getenv("SILICONFLOW_API_KEY", "your-api-key-here")

# 模型与 Agent
llm = ChatOpenAI(
    model="THUDM/glm-4-9b-chat",
    temperature=0,
    max_retries=3,
    api_key=api_key,
    base_url="https://api.siliconflow.cn/v1",
)
agent = create_react_agent(llm, tools, state_modifier=SYSTEM_PROMPT)

# ==========================================
# 3. 运行测试
# ==========================================
if __name__ == "__main__":
    # 为了测试正常运行，请在同级目录下放一个名为 knowledge_base.pdf 的文件
    # 并且在终端 export SILICONFLOW_API_KEY="sk-..."
    question = "请基于知识库，概述 RAG 的优缺点，并给出引用。"
    print(f"User: {question}\n")
    print("Agent thinking...\n")
    
    result = agent.invoke({"messages": [("user", question)]})
    final_answer = result["messages"][-1].content
    print("最终答案:\n", final_answer)